# Comparação das soluções para a aorta

Compara três estratégias promissoras nas mesmas imagens de treino e validação:

- **Baseline:** level set fixo sem filtro robusto ou envelope da trajetória;
- **Filtro + envelope:** filtro robusto dos círculos, cinco círculos sintéticos e envelope de `2.25r`, mantendo o level set original;
- **Filtro + envelope + level set refinado:** mesma correção geométrica com `balloon=0.6`, inicialização em `0.10r` e 26 iterações.

As variantes de cobertura fixa e o controlador adaptativo antigo foram retirados porque não melhoraram o resultado. A inspeção manual classifica somente a qualidade da máscara da aorta; o sucesso dos óstios e o Dice são carregados dos CSVs de cada run.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Localiza a raiz antes de importar os módulos do projeto.
current = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_visual_reviews,
    resolve_aorta_review_summary_path,
)
from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 40)

## 1. Configuração e carregamento

Os caminhos dos runs e as classificações visuais ficam centralizados em `config/aorta_visual_reviews.json`. A comparação usa apenas variantes disponíveis nas mesmas coortes de 30 imagens de treino e 60 de validação.


In [2]:
REVIEW_CONFIG_PATH = REPO_ROOT / "config/aorta_visual_reviews.json"
SPLITS = ("train", "val")
VARIANTS = (
    "normal",
    "filter_envelope_current",
    "levelset_b0_6_r0_10_i26",
)

SPLIT_NAMES = {"train": "Treino", "val": "Validação"}
DISPLAY_NAMES = {
    "normal": "Baseline",
    "filter_envelope_current": "Filtro + envelope",
    "levelset_b0_6_r0_10_i26": "Filtro + envelope + level set refinado",
}
SUCCESS_STATUSES = {
    "both correct",
    "both tolerable",
    "both ostia correct",
    "both ostia tolerable",
}
SUMMARY_COLUMNS = [
    "IMG_ID",
    "artery_dice",
    "ostia_detection_status",
    "aorta_mask_voxel_count",
    "aorta_volume_fraction",
]

review_catalog = load_aorta_visual_reviews(REVIEW_CONFIG_PATH)
reviews = {
    (split, variant): get_aorta_visual_review(review_catalog, variant, split)
    for split in SPLITS
    for variant in VARIANTS
}


def load_solution(split, variant):
    """Carrega métricas automáticas e acrescenta o rótulo visual da aorta."""
    review = reviews[(split, variant)]
    summary_path = resolve_aorta_review_summary_path(REPO_ROOT, review, split)
    frame = pd.read_csv(summary_path)

    missing_columns = set(SUMMARY_COLUMNS).difference(frame.columns)
    if missing_columns:
        raise ValueError(
            f"Colunas ausentes em {variant}/{split}: {sorted(missing_columns)}"
        )

    # Mantém somente as métricas compartilhadas pelos três experimentos.
    frame = frame[SUMMARY_COLUMNS].copy()
    frame["IMG_ID"] = pd.to_numeric(frame["IMG_ID"], errors="raise").astype(int)

    expected_ids = review["aorta_good_ids"] | review["aorta_bad_ids"]
    observed_ids = set(frame["IMG_ID"])
    if observed_ids != expected_ids:
        raise ValueError(
            f"IDs incompatíveis em {variant}/{split}: "
            f"ausentes={sorted(expected_ids - observed_ids)}; "
            f"não revisados={sorted(observed_ids - expected_ids)}"
        )

    # Both correct e both tolerable representam sucesso dos dois óstios.
    normalized_status = (
        frame["ostia_detection_status"]
        .astype(str)
        .str.lower()
        .str.replace("_", " ", regex=False)
        .str.strip()
    )
    frame["ostia_success"] = normalized_status.isin(SUCCESS_STATUSES)
    frame["aorta_visual_good"] = frame["IMG_ID"].isin(review["aorta_good_ids"])
    frame["variant"] = variant
    frame["split"] = split
    return frame


solutions = {
    (split, variant): load_solution(split, variant)
    for split in SPLITS
    for variant in VARIANTS
}

for split in SPLITS:
    cohort_ids = [set(solutions[(split, variant)]["IMG_ID"]) for variant in VARIANTS]
    if any(ids != cohort_ids[0] for ids in cohort_ids[1:]):
        raise ValueError(f"As variantes de {split} não usam a mesma coorte.")
    print(f"{SPLIT_NAMES[split]}: {len(cohort_ids[0])} exames em cada variante")


Treino: 30 exames em cada variante
Validação: 60 exames em cada variante


## 2. Resultados gerais

A tabela reúne qualidade visual da aorta, sucesso automático dos dois óstios e Dice da segmentação arterial. O volume médio é mantido como apoio para identificar alterações globais na máscara.


In [3]:
overview_rows = []
for split in SPLITS:
    for variant in VARIANTS:
        frame = solutions[(split, variant)]
        overview_rows.append(
            {
                "subconjunto": SPLIT_NAMES[split],
                "solução": DISPLAY_NAMES[variant],
                "imagens": len(frame),
                "aortas_boas": int(frame["aorta_visual_good"].sum()),
                "aortas_boas_%": 100 * frame["aorta_visual_good"].mean(),
                "sucesso_óstios_csv": int(frame["ostia_success"].sum()),
                "sucesso_óstios_%": 100 * frame["ostia_success"].mean(),
                "dice_médio": frame["artery_dice"].mean(),
                "dice_mediano": frame["artery_dice"].median(),
                "volume_aorta_médio_%": 100 * frame["aorta_volume_fraction"].mean(),
            }
        )

overview_df = pd.DataFrame(overview_rows)
display(overview_df.round(4))

,subconjunto,solução,imagens,aortas_boas,aortas_boas_%,sucesso_óstios_csv,sucesso_óstios_%,dice_médio,dice_mediano,volume_aorta_médio_%
0,Treino,Baseline,30,23,76.6667,27,90.0000,0.6148,0.6311,1.5545
1,Treino,Filtro + envelope,30,29,96.6667,28,93.3333,0.6237,0.6389,1.2439
2,Treino,Filtro + envelope + level set refinado,30,30,100.0000,26,86.6667,0.5826,0.6233,1.1715
3,Validação,Baseline,60,52,86.6667,48,80.0000,0.5650,0.6364,1.3917
4,Validação,Filtro + envelope,60,52,86.6667,50,83.3333,0.5858,0.6364,1.2397
5,Validação,Filtro + envelope + level set refinado,60,56,93.3333,51,85.0000,0.5851,0.6272,1.1042


## 3. Diferença em relação ao baseline

Os deltas são calculados de forma pareada pelos mesmos `IMG_IDs`. Valores positivos em Dice e sucesso dos óstios favorecem a variante. As listas de aortas corrigidas e pioradas vêm exclusivamente da inspeção visual.


In [4]:
comparison_rows = []
for split in SPLITS:
    baseline = solutions[(split, "normal")].set_index("IMG_ID").sort_index()
    baseline_review = reviews[(split, "normal")]

    for variant in VARIANTS[1:]:
        candidate = solutions[(split, variant)].set_index("IMG_ID").sort_index()
        candidate_review = reviews[(split, variant)]
        paired_ids = baseline.index.intersection(candidate.index)

        normal_bad = baseline_review["aorta_bad_ids"]
        candidate_bad = candidate_review["aorta_bad_ids"]
        comparison_rows.append(
            {
                "subconjunto": SPLIT_NAMES[split],
                "solução": DISPLAY_NAMES[variant],
                "delta_dice_médio": (
                    candidate.loc[paired_ids, "artery_dice"]
                    - baseline.loc[paired_ids, "artery_dice"]
                ).mean(),
                "delta_sucesso_óstios_pp": 100 * (
                    candidate.loc[paired_ids, "ostia_success"].mean()
                    - baseline.loc[paired_ids, "ostia_success"].mean()
                ),
                "delta_aortas_boas": (
                    int(candidate.loc[paired_ids, "aorta_visual_good"].sum())
                    - int(baseline.loc[paired_ids, "aorta_visual_good"].sum())
                ),
                "máscaras_com_voxels_diferentes": int(
                    (
                        candidate.loc[paired_ids, "aorta_mask_voxel_count"]
                        != baseline.loc[paired_ids, "aorta_mask_voxel_count"]
                    ).sum()
                ),
                "aortas_corrigidas": sorted(normal_bad - candidate_bad),
                "aortas_pioradas": sorted(candidate_bad - normal_bad),
            }
        )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

,subconjunto,solução,delta_dice_médio,delta_sucesso_óstios_pp,delta_aortas_boas,máscaras_com_voxels_diferentes,aortas_corrigidas,aortas_pioradas
0,Treino,Filtro + envelope,0.0089,3.3333,6,20,"[44, 175, 330, 608, 752, 760]",[]
1,Treino,Filtro + envelope + level set refinado,-0.0322,-3.3333,7,30,"[44, 175, 330, 603, 608, 752, 760]",[]
2,Validação,Filtro + envelope,0.0208,3.3333,0,39,[],[]
3,Validação,Filtro + envelope + level set refinado,0.0202,5.0000,4,60,"[134, 444, 597, 602]",[]


## 4. Comparação visual das métricas

Cada linha representa um subconjunto. As escalas percentuais da qualidade da aorta e dos óstios são separadas do Dice para evitar interpretações equivocadas.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
colors = ["#4C78A8", "#54A24B", "#E45756"]
metrics = (
    ("aortas_boas_%", "Aortas visualmente boas (%)", (0, 110)),
    ("sucesso_óstios_%", "Sucesso dos óstios pelo CSV (%)", (0, 110)),
    ("dice_médio", "Dice médio", (0, 1.08)),
)

for row, split in enumerate(SPLITS):
    split_df = overview_df[overview_df["subconjunto"].eq(SPLIT_NAMES[split])]
    for column, (metric, title, limits) in enumerate(metrics):
        axis = axes[row, column]
        bars = axis.bar(split_df["solução"], split_df[metric], color=colors)
        axis.set_title(f"{SPLIT_NAMES[split]}: {title}", fontsize=11)
        axis.set_ylim(*limits)
        axis.tick_params(axis="x", rotation=18, labelsize=9)
        axis.grid(axis="y", alpha=0.25)

        decimals = 3 if metric == "dice_médio" else 1
        labels = [f"{value:.{decimals}f}" for value in split_df[metric]]
        axis.bar_label(bars, labels=labels, padding=3, fontsize=9)

plt.show()


## 5. Síntese

A melhor configuração pode variar conforme o desfecho. Esta síntese informa separadamente o maior Dice, a melhor qualidade visual da aorta e a maior taxa de sucesso dos óstios em cada subconjunto.


In [6]:
for split in SPLITS:
    split_df = overview_df[overview_df["subconjunto"].eq(SPLIT_NAMES[split])]
    best_dice = split_df.loc[split_df["dice_médio"].idxmax()]
    best_aorta = split_df.loc[split_df["aortas_boas_%"].idxmax()]
    best_ostia = split_df.loc[split_df["sucesso_óstios_%"].idxmax()]

    print(SPLIT_NAMES[split])
    print(f"- Maior Dice: {best_dice['solução']} ({best_dice['dice_médio']:.4f})")
    print(
        f"- Mais aortas visualmente boas: {best_aorta['solução']} "
        f"({best_aorta['aortas_boas_%']:.1f}%)"
    )
    print(
        f"- Maior sucesso dos óstios: {best_ostia['solução']} "
        f"({best_ostia['sucesso_óstios_%']:.1f}%)"
    )

Treino
- Maior Dice: Filtro + envelope (0.6237)
- Mais aortas visualmente boas: Filtro + envelope + level set refinado (100.0%)
- Maior sucesso dos óstios: Filtro + envelope (93.3%)
Validação
- Maior Dice: Filtro + envelope (0.5858)
- Mais aortas visualmente boas: Filtro + envelope + level set refinado (93.3%)
- Maior sucesso dos óstios: Filtro + envelope + level set refinado (85.0%)
